# LinkML Notebook

The idea of this notebook is to create an environemnt that allows to rapidly iterate over different LinkML schemas and data files to check the different features of LinkML and how they work in practice. The notebook is not meant to be a tutorial on how to use LinkML, but rather a playground for testing different features and seeing how they work in practice.

The model and the data are defined in two seperate cells each with a yaml magic command that saves the content of the cell as python dictionary.

In [1]:
# register the yaml magic command

%reload_ext yamlmagic

## Helper Code

In [2]:
from linkml_runtime.linkml_model import SchemaDefinition
from linkml_runtime.utils.schemaview import SchemaView
from linkml_runtime.loaders import YAMLLoader
from linkml_runtime.dumpers import RDFLibDumper
from linkml_runtime.dumpers import JSONDumper
from linkml.generators.pythongen import PythonGenerator

import types

def process_linkml(schema, data, target_class_name):
    
    # load schema and create a schemaview
    schema = YAMLLoader().load(schema, target_class=SchemaDefinition)
    view = SchemaView(schema)

    # make real python classes from the schema
    python_code = PythonGenerator(schema).serialize()

    # create a dynamic module and execute the generated code in it
    module = types.ModuleType("my_schema")
    exec(python_code, module.__dict__)

    # load the data into an instance of a class defined in the schema
    data = YAMLLoader().load(data, target_class=getattr(module, target_class_name))

    # convert the data to RDF and print it
    print("RDF:\n")
    print(RDFLibDumper().dumps(data, schemaview=view))

    # convert the data to JSON and print it
    print("\nJSON:\n")
    print(JSONDumper().dumps(data))
    

/Volumes/CaseSensitive/political-affairs-ech-group/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Basic Model

see: https://linkml.io/linkml/intro/tutorial01.html

In [3]:
%%yaml schema

id: https://ld.ech.ch/tutorial
prefixes:
  linkml: https://w3id.org/linkml/
  tut: https://ld.ech.ch/tutorial/
imports:
  - linkml:types
default_range: string
default_prefix: tut

classes:
  Person:
    attributes:
      id:
      full_name:
      aliases:
      phone:
      age:

<IPython.core.display.Javascript object>

In [4]:
%%yaml data

id: ORCID:1234
full_name: Clark Kent
age: "32"
phone: 555-555-5555

<IPython.core.display.Javascript object>

In [5]:
process_linkml(schema, data, "Person")

schema.items()

RDF:

@prefix tut: <https://ld.ech.ch/tutorial/> .

[] a tut:Person ;
    tut:age "32" ;
    tut:full_name "Clark Kent" ;
    tut:id "ORCID:1234" ;
    tut:phone "555-555-5555" .



JSON:

{
  "id": "ORCID:1234",
  "full_name": "Clark Kent",
  "phone": "555-555-5555",
  "age": "32",
  "@type": "Person"
}


dict_items([('id', 'https://ld.ech.ch/tutorial'), ('prefixes', {'linkml': 'https://w3id.org/linkml/', 'tut': 'https://ld.ech.ch/tutorial/'}), ('imports', ['linkml:types']), ('default_range', 'string'), ('default_prefix', 'tut'), ('classes', {'Person': {'attributes': {'id': None, 'full_name': None, 'aliases': None, 'phone': None, 'age': None}}})])

## References and Imports

In [6]:
%%yaml schema

id: https://ld.ech.ch/tutorial
prefixes:
  xsd: http://www.w3.org/2001/XMLSchema#
  prov: http://www.w3.org/ns/prov#
  dcterms: http://purl.org/dc/terms/
  linkml: https://w3id.org/linkml/
  schema: http://schema.org/
  tut: https://ld.ech.ch/tutorial/
imports:
  - linkml:types
  - input/schema_dates
default_prefix: tut
default_range: string
  
classes:
  
  Reference:
    description: A reference to another entity, e.g. a person, organization, or location. This class can be used as a base class for more specific reference types. This class is not meant to be instantiated directly, but serves as a base for other reference classes.
    abstract: true
    slots:
      - id
      - datetime_created

  PersonReference:
    is_a: Reference
    description: A reference to a person, e.g. a politician, a member of parliament, or a government official.
    slots:
      - name
      - party
      - position

  Container:
    description: A container for references.
    tree_root: true
    slots:
      - id
      - person_references

slots:

  id:
    identifier: true
    required: true

  name:
    slot_uri: schema:name

  party:
    slot_uri: tut:party

  position:
    slot_uri: tut:position

  person_references:
    slot_uri: tut:personReference
    range: PersonReference
    multivalued: true
    inlined_as_list: true


<IPython.core.display.Javascript object>

In [7]:
%%yaml data

id: tut:references-1
person_references:
  - id: tut:p1
    name: John Doe
    party: Independent
    position: Member of Parliament
  - id: tut:p2
    name: Jane Smith
    party: Green Party
    position: Minister of Environment

<IPython.core.display.Javascript object>

In [8]:
process_linkml(schema, data, "Container")

RDF:

@prefix schema1: <http://schema.org/> .
@prefix tut: <https://ld.ech.ch/tutorial/> .

tut:references-1 a tut:Container ;
    tut:personReference tut:p1,
        tut:p2 .

tut:p1 a tut:PersonReference ;
    schema1:name "John Doe" ;
    tut:party "Independent" ;
    tut:position "Member of Parliament" .

tut:p2 a tut:PersonReference ;
    schema1:name "Jane Smith" ;
    tut:party "Green Party" ;
    tut:position "Minister of Environment" .



JSON:

{
  "id": "tut:references-1",
  "person_references": [
    {
      "id": "tut:p1",
      "name": "John Doe",
      "party": "Independent",
      "position": "Member of Parliament"
    },
    {
      "id": "tut:p2",
      "name": "Jane Smith",
      "party": "Green Party",
      "position": "Minister of Environment"
    }
  ],
  "@type": "Container"
}


## Mix Ins

In [9]:
%%yaml schema

id: https://ld.ech.ch/tutorial
prefixes:
  xsd: http://www.w3.org/2001/XMLSchema#
  prov: http://www.w3.org/ns/prov#
  dcterms: http://purl.org/dc/terms/
  linkml: https://w3id.org/linkml/
  schema: http://schema.org/
  tut: https://ld.ech.ch/tutorial/
imports:
  - linkml:types
default_prefix: tut
default_range: string
  
classes:

  Container:
    description: A container for references.
    tree_root: true
    slots:
      - id
      - memberships

  HasDuration:
    description: A mixin class that adds duration slots to a class.
    mixin: true
    slots:
      - valid_from
      - valid_through
      - is_current

  Membership:
    description: A class that represents a membership in an organization, e.g. a political party or a government.
    mixins: 
      - HasDuration
    slots:
      - id
      - name

slots:

  id:
    identifier: true
    required: true

  name:
    slot_uri: schema:name

  memberships:
    slot_uri: tut:membership
    range: Membership
    multivalued: true
    inlined_as_list: true

  valid_from:
    slot_uri: schema:validFrom
    range: date

  valid_through:
    slot_uri: schema:validThrough
    range: date

  is_current:
    slot_uri: tut:isCurrent
    range: boolean


<IPython.core.display.Javascript object>

In [10]:
%%yaml data

id: tut:membership-1
memberships:
  - id: tut:m1
    name: Federal Council
    valid_from: 2020-01-01
    valid_through: 2024-12-31
    is_current: false
  - id: tut:m2
    name: Mayor of Zurich
    valid_from: 2020-01-01
    is_current: true

<IPython.core.display.Javascript object>

In [11]:
process_linkml(schema, data, "Container")

RDF:

@prefix schema1: <http://schema.org/> .
@prefix tut: <https://ld.ech.ch/tutorial/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

tut:membership-1 a tut:Container ;
    tut:membership tut:m1,
        tut:m2 .

tut:m1 a tut:Membership ;
    schema1:name "Federal Council" ;
    schema1:validFrom "2020-01-01"^^xsd:date ;
    schema1:validThrough "2024-12-31"^^xsd:date ;
    tut:isCurrent false .

tut:m2 a tut:Membership ;
    schema1:name "Mayor of Zurich" ;
    schema1:validFrom "2020-01-01"^^xsd:date ;
    tut:isCurrent true .



JSON:

{
  "id": "tut:membership-1",
  "memberships": [
    {
      "id": "tut:m1",
      "name": "Federal Council",
      "valid_from": "2020-01-01",
      "valid_through": "2024-12-31",
      "is_current": false
    },
    {
      "id": "tut:m2",
      "name": "Mayor of Zurich",
      "valid_from": "2020-01-01",
      "is_current": true
    }
  ],
  "@type": "Container"
}


## Type Designators

In [12]:
%%yaml schema

id: https://example.org/animal-schema
name: animal-schema
prefixes:
  linkml: https://w3id.org/linkml/
  rdf: http://www.w3.org/1999/02/22-rdf-syntax-ns#
  ex: https://example.org/
default_prefix: ex
default_range: string

imports:
  - linkml:types

classes:

  Animal:
    description: Base class for all animals
    slots:
      - id
      - name
      - animal_type

  Dog:
    is_a: Animal
    description: A dog
    slots:
      - breed

  Cat:
    is_a: Animal
    description: A cat
    slots:
      - indoor

  AnimalCollection:
    tree_root: true
    slots:
      - animals

slots:

  id:
    identifier: true
    range: string
  name:
    range: string
  animal_type:
    designates_type: true
    slot_uri: rdf:type
    range: uriorcurie
    required: true
  breed:
    range: string
  indoor:
    range: boolean
  animals:
    range: Animal
    multivalued: true
    inlined_as_list: true

<IPython.core.display.Javascript object>

In [13]:
%%yaml data

animals:
  - id: ex:a001
    name: Rex
    animal_type: ex:Dog
    breed: Labrador

  - id: ex:a002
    name: Whiskers
    animal_type: ex:Cat
    indoor: true

  - id: ex:a003
    name: Buddy
    animal_type: ex:Dog
    breed: Beagle

<IPython.core.display.Javascript object>

In [14]:
process_linkml(schema, data, "AnimalCollection")

RDF:

@prefix ex: <https://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:a001 a "ex:Dog"^^xsd:anyURI ;
    ex:breed "Labrador" ;
    ex:name "Rex" .

ex:a002 a "ex:Cat"^^xsd:anyURI ;
    ex:indoor true ;
    ex:name "Whiskers" .

ex:a003 a "ex:Dog"^^xsd:anyURI ;
    ex:breed "Beagle" ;
    ex:name "Buddy" .

[] a ex:AnimalCollection ;
    ex:animals ex:a001,
        ex:a002,
        ex:a003 .



JSON:

{
  "animals": [
    {
      "id": "ex:a001",
      "animal_type": "ex:Dog",
      "name": "Rex",
      "breed": "Labrador"
    },
    {
      "id": "ex:a002",
      "animal_type": "ex:Cat",
      "name": "Whiskers",
      "indoor": true
    },
    {
      "id": "ex:a003",
      "animal_type": "ex:Dog",
      "name": "Buddy",
      "breed": "Beagle"
    }
  ],
  "@type": "AnimalCollection"
}


Funktioniert noch nicht so richtig, sollte im RDF so aussehen:

```turtle
@prefix ex: <https://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:a001 a ex:Dog ;
    ex:breed "Labrador" ;
    ex:name "Rex" .
```

## Any Of

In [15]:
%%yaml schema

id: https://ld.ech.ch/tutorial
prefixes:
  xsd: http://www.w3.org/2001/XMLSchema#
  prov: http://www.w3.org/ns/prov#
  dcterms: http://purl.org/dc/terms/
  linkml: https://w3id.org/linkml/
  schema: http://schema.org/
  tut: https://ld.ech.ch/tutorial/
imports:
  - linkml:types
default_prefix: tut
default_range: string
  
classes:

    Container:
        description: A container for persons.
        tree_root: true
        slots:
            - id
            - persons

    Person:
        slots:
            - id 
            - name

    TimedName:
        slots:
            - name_string
            - valid_from
            - valid_through

    Any:
        class_uri: linkml:Any


slots:

    id:
        identifier: true
        required: true        

    persons:
        slot_uri: tut:person
        range: Person
        multivalued: true
        inlined_as_list: true

    name:
        #range: Any
        any_of:
            - range: string
            - range: TimedName

    name_string:
        range: string

    valid_from:
        range: date

    valid_through:
        range: date

<IPython.core.display.Javascript object>

In [16]:
%%yaml data

id: tut:person-container-1
persons:
    - id: tut:p1
      name: John Doe
    - id: tut:p2
      name:
        name_string: Jane Smith
        valid_from: 2020-01-01

<IPython.core.display.Javascript object>

In [17]:
process_linkml(schema, data, "Container")

RDF:

@prefix tut: <https://ld.ech.ch/tutorial/> .

tut:person-container-1 a tut:Container ;
    tut:person tut:p1,
        tut:p2 .

tut:p1 a tut:Person ;
    tut:name "John Doe" .

tut:p2 a tut:Person ;
    tut:name "JsonObj(name_string='Jane Smith', valid_from=datetime.date(2020, 1, 1))" .



JSON:

{
  "id": "tut:person-container-1",
  "persons": [
    {
      "id": "tut:p1",
      "name": "John Doe"
    },
    {
      "id": "tut:p2",
      "name": "JsonObj(name_string='Jane Smith', valid_from=datetime.date(2020, 1, 1))"
    }
  ],
  "@type": "Container"
}


Funktioniert nur so mehr schlecht als recht, insbesondere die RDF Serialisierung ist nicht korrekt, besser Problem so lösen, wie im nächsten Abschnitt beschrieben.

## Enum/Named_String/String

Idee: Eine Datenstruktur ermöglichen, die sowohl einfache Strings, multilingual Strings als auch Enums erlaubt. Und diese Enums auch noch mit multilingualen Labels versehen können

In [18]:
%%yaml schema

id: https://ld.ech.ch/tutorial
prefixes:
  xsd: http://www.w3.org/2001/XMLSchema#
  prov: http://www.w3.org/ns/prov#
  dcterms: http://purl.org/dc/terms/
  linkml: https://w3id.org/linkml/
  schema: http://schema.org/
  tut: https://ld.ech.ch/tutorial/
imports:
  - linkml:types
default_prefix: tut
default_range: string
  
classes:

    Person:
        tree_root: true
        slots:
            - id
            - name
    
    Name:
        slots:
            - name_enum
            - multilingual_value
            - label

    MultilingualValue:
        slots:
            - language
            - value

    NameEnum:
        slots:
            - name_enum_type
            - multilingual_value
            - value

    
slots:

    id:
        identifier: true
        required: true        

    name:
        range: Name

    name_enum:
        range: NameEnum
        multivalued: true
        inlined_as_list: true

    multilingual_value:
        range: MultilingualValue
        multivalued: true
        inlined_as_list: true

    label:
        multivalued: true
        inlined_as_list: true

    language:

    value:

    name_enum_type:
        range: NameEnumType
    

enums:

    NameEnumType:
        permissible_values:
            surname:
                meaning: tut:Surname
            name:
                meaning: tut:Name


<IPython.core.display.Javascript object>

### Nur `label`

In [19]:
%%yaml data
id: tut:p1
name:
    label: John Doe

<IPython.core.display.Javascript object>

In [20]:
process_linkml(schema, data, "Person")

RDF:

@prefix tut: <https://ld.ech.ch/tutorial/> .

tut:p1 a tut:Person ;
    tut:name [ a tut:Name ;
            tut:label "John Doe" ] .



JSON:

{
  "id": "tut:p1",
  "name": {
    "label": [
      "John Doe"
    ]
  },
  "@type": "Person"
}


### Nur `multilingual_value`

In [21]:
%%yaml data
id: tut:p1
name:
    multilingual_value:
        - language: en
          value: John Doe
        - language: de
          value: Johann Doe

<IPython.core.display.Javascript object>

In [22]:
process_linkml(schema, data, "Person")

RDF:

@prefix tut: <https://ld.ech.ch/tutorial/> .

tut:p1 a tut:Person ;
    tut:name [ a tut:Name ;
            tut:multilingual_value [ a tut:MultilingualValue ;
                    tut:language "de" ;
                    tut:value "Johann Doe" ],
                [ a tut:MultilingualValue ;
                    tut:language "en" ;
                    tut:value "John Doe" ] ] .



JSON:

{
  "id": "tut:p1",
  "name": {
    "multilingual_value": [
      {
        "language": "en",
        "value": "John Doe"
      },
      {
        "language": "de",
        "value": "Johann Doe"
      }
    ]
  },
  "@type": "Person"
}


### Nur `name_enum`

In [23]:
%%yaml data
id: tut:p1
name:
    name_enum:
        - name_enum_type: surname
          value: John
        - name_enum_type: name
          value: Doe

<IPython.core.display.Javascript object>

In [24]:
process_linkml(schema, data, "Person")

RDF:

@prefix tut: <https://ld.ech.ch/tutorial/> .

tut:p1 a tut:Person ;
    tut:name [ a tut:Name ;
            tut:name_enum [ a tut:NameEnum ;
                    tut:name_enum_type tut:Surname ;
                    tut:value "John" ],
                [ a tut:NameEnum ;
                    tut:name_enum_type tut:Name ;
                    tut:value "Doe" ] ] .



JSON:

{
  "id": "tut:p1",
  "name": {
    "name_enum": [
      {
        "name_enum_type": "surname",
        "value": "John"
      },
      {
        "name_enum_type": "name",
        "value": "Doe"
      }
    ]
  },
  "@type": "Person"
}


### `name_enum` mit `multilingual_value`

In [25]:
%%yaml data

id: tut:p1
name:
    name_enum:
        - name_enum_type: surname
          multilingual_value:
            - language: en
              value: John
            - language: de
              value: Johannes

<IPython.core.display.Javascript object>

In [26]:
process_linkml(schema, data, "Person")

RDF:

@prefix tut: <https://ld.ech.ch/tutorial/> .

tut:p1 a tut:Person ;
    tut:name [ a tut:Name ;
            tut:name_enum [ a tut:NameEnum ;
                    tut:multilingual_value [ a tut:MultilingualValue ;
                            tut:language "en" ;
                            tut:value "John" ],
                        [ a tut:MultilingualValue ;
                            tut:language "de" ;
                            tut:value "Johannes" ] ;
                    tut:name_enum_type tut:Surname ] ] .



JSON:

{
  "id": "tut:p1",
  "name": {
    "name_enum": [
      {
        "name_enum_type": "surname",
        "multilingual_value": [
          {
            "language": "en",
            "value": "John"
          },
          {
            "language": "de",
            "value": "Johannes"
          }
        ]
      }
    ]
  },
  "@type": "Person"
}


### to do

Wie müsste es aussehen, wenn die Enumeration noch ein "other" hat und das noch in einem separaten slot beschrieben werden soll (auch multilingual)?